In [ ]:
"""
Mental Health Text Classification — TF-IDF Pipeline
====================================================
Semantic blocks (each block can be copy-pasted into a Jupyter notebook cell):

  Block 1 — Imports & Config
  Block 2 — Load & Select Columns
  Block 3 — Text Filtering (noise reduction)
  Block 4 — Stopword Removal & Stemming
  Block 5 — Train / Test Split (80/20 stratified)
  Block 6 — TF-IDF Feature Extraction
  Block 7 — Classification & Evaluation
"""

# ══════════════════════════════════════════════════════════════════════════════
# Block 1 — Imports & Config
# ══════════════════════════════════════════════════════════════════════════════
import re
import warnings

import nltk
import numpy as np
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings('ignore')

tqdm.pandas(desc='processing')
nltk.download('stopwords', quiet=True)

from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_PATH = 'data_filtered.csv'

# ── Filtering thresholds ───────────────────────────────────────────────────
MIN_WORDS    = 3      # minimum token count after cleaning (drop very short texts)
MAX_WORDS    = 400    # truncate very long texts to this many tokens
MIN_CHAR_LEN = 20     # drop texts shorter than this many characters (raw)

# ── Classes to drop (ambiguous / too noisy) ────────────────────────────────
CLASSES_TO_DROP = ['тревожное р-во/депрессия', 'паранойя', 'тревожное р-во/невроз']

# ── Split ──────────────────────────────────────────────────────────────────
TEST_SIZE    = 0.20
RANDOM_STATE = 42

# ── TF-IDF — change these parameters to tune noise reduction ──────────────
TFIDF_PARAMS = dict(
    ngram_range = (1, 2),      # unigrams + bigrams
    max_features= 50000,     # vocabulary cap
    sublinear_tf= True,        # apply 1 + log(tf) scaling
    min_df      = 5,           # ignore terms appearing in fewer than N docs
    max_df      = 0.85,        # ignore terms appearing in more than 95 % of docs
    token_pattern= r'[а-яёa-z]{2,}',  # only Cyrillic/Latin words ≥ 2 chars
)

# ── NLP tools ──────────────────────────────────────────────────────────────
RU_STOPWORDS = set(stopwords.words('russian'))
STEMMER      = SnowballStemmer('russian')

np.random.seed(RANDOM_STATE)

print("=" * 64)
print("MENTAL HEALTH CLASSIFICATION — TF-IDF Pipeline")
print(f"Data:        {DATA_PATH}")
print(f"Min words:   {MIN_WORDS}  |  Max words: {MAX_WORDS}")
print(f"Test size:   {TEST_SIZE}")
print(f"TF-IDF ngram:{TFIDF_PARAMS['ngram_range']}  "
      f"max_features={TFIDF_PARAMS['max_features']}  "
      f"min_df={TFIDF_PARAMS['min_df']}  "
      f"max_df={TFIDF_PARAMS['max_df']}")
print("=" * 64)

MENTAL HEALTH CLASSIFICATION — TF-IDF Pipeline
Data:        data_filtered.csv
Min words:   3  |  Max words: 400
Test size:   0.2
TF-IDF ngram:(1, 2)  max_features=50000  min_df=5  max_df=0.85


In [24]:

# ══════════════════════════════════════════════════════════════════════════════
# Block 2 — Load & Select Columns
# ══════════════════════════════════════════════════════════════════════════════
print("\n[1/6] Loading data and selecting columns...")

df = pd.read_csv(DATA_PATH, sep=';', encoding='utf-8')
print(f"Raw shape: {df.shape}  |  columns: {list(df.columns)}")

# Keep only the two required columns
df = df[['text', 'tag']].copy()
df = df.dropna(subset=['text', 'tag']).reset_index(drop=True)
df['text'] = df['text'].astype(str)
df['tag']  = df['tag'].astype(str).str.strip()

print(f"After column selection & dropna: {len(df):,} rows, {df['tag'].nunique()} classes")

# Drop ambiguous / noisy classes
before = len(df)
df = df[~df['tag'].isin(CLASSES_TO_DROP)].copy().reset_index(drop=True)
print(f"After dropping ambiguous classes: removed {before - len(df):,} → {len(df):,} rows, "
      f"{df['tag'].nunique()} classes")

print("\nClass distribution (raw):")
print(df['tag'].value_counts().to_string())



[1/6] Loading data and selecting columns...
Raw shape: (31402, 5)  |  columns: ['url', 'date', 'text', 'tag', 'source']
After column selection & dropna: 31,402 rows, 9 classes
After dropping ambiguous classes: removed 1,108 → 30,294 rows, 6 classes

Class distribution (raw):
tag
депрессия         16712
тревожное р-во     5875
ОКР                2824
ПРЛ                2738
БАР                1397
шизофрения          748


In [25]:


# ══════════════════════════════════════════════════════════════════════════════
# Block 3 — Text Filtering (noise reduction)
# ══════════════════════════════════════════════════════════════════════════════
print("\n[2/6] Applying text filters...")

# Pre-compiled regex patterns
_URL_RE    = re.compile(r'https?://\S+|www\.\S+')
_EMAIL_RE  = re.compile(r'\S+@\S+\.\S+')
_HTML_RE   = re.compile(r'<[^>]+>')
_MENTION_RE= re.compile(r'@\w+')           # @username mentions
_HASHTAG_RE= re.compile(r'#\w+')           # #hashtags
_DIGIT_RE  = re.compile(r'\b\d+\b')        # standalone numbers
_PUNCT_RE  = re.compile(r'[^а-яёА-ЯЁa-zA-Z\s]')  # keep only letters + spaces
_WS_RE     = re.compile(r'\s+')


def filter_text(text: str) -> str:
    """
    Multi-step noise reduction pipeline.
    Returns a lowercased, cleaned string (stopwords still present at this stage).
    """
    # 1. Remove URLs
    text = _URL_RE.sub(' ', text)
    # 2. Remove e-mail addresses
    text = _EMAIL_RE.sub(' ', text)
    # 3. Strip HTML tags
    text = _HTML_RE.sub(' ', text)
    # 4. Remove @mentions and #hashtags
    text = _MENTION_RE.sub(' ', text)
    text = _HASHTAG_RE.sub(' ', text)
    # 5. Remove standalone digit tokens (years, phone fragments, etc.)
    text = _DIGIT_RE.sub(' ', text)
    # 6. Remove punctuation / special characters — keep only Cyrillic/Latin letters
    text = _PUNCT_RE.sub(' ', text)
    # 7. Lowercase
    text = text.lower()
    # 8. Collapse whitespace
    text = _WS_RE.sub(' ', text).strip()
    # 9. Truncate to MAX_WORDS tokens
    tokens = text.split()
    if len(tokens) > MAX_WORDS:
        tokens = tokens[:MAX_WORDS]
    return ' '.join(tokens)


tqdm.pandas(desc='filter_text')
df['text_clean'] = df['text'].progress_apply(filter_text)

# Drop texts that are too short even before stopword removal
before = len(df)
df = df[df['text_clean'].str.len() >= MIN_CHAR_LEN].copy().reset_index(drop=True)
print(f"After min-char filter ({MIN_CHAR_LEN}): removed {before - len(df):,} → {len(df):,} kept")

print(f"Sample cleaned text:\n  {df['text_clean'].iloc[0][:120]}...")




[2/6] Applying text filters...


filter_text: 100%|██████████| 30294/30294 [00:01<00:00, 28484.93it/s]

After min-char filter (20): removed 87 → 30,207 kept
Sample cleaned text:
  всем добрый день у меня бар типа начала часто употреблять спиртные напитки при этом не помню что происходило именно из з...


In [26]:

# ══════════════════════════════════════════════════════════════════════════════
# Block 4 — Stopword Removal & Stemming
# ══════════════════════════════════════════════════════════════════════════════
print("\n[3/6] Removing Russian stopwords and applying Snowball stemming...")


def remove_stopwords_and_stem(text: str) -> str:
    """
    1. Split into tokens.
    2. Drop Russian stopwords.
    3. Apply Snowball (Porter-like) stemmer for Russian.
    4. Drop tokens shorter than 2 characters (stemming artefacts).
    """
    tokens = text.split()
    tokens = [t for t in tokens if t not in RU_STOPWORDS]
    tokens = [STEMMER.stem(t) for t in tokens]
    tokens = [t for t in tokens if len(t) >= 2]
    return ' '.join(tokens)


tqdm.pandas(desc='stopwords+stem')
df['text_processed'] = df['text_clean'].progress_apply(remove_stopwords_and_stem)

# Drop texts that became too short after stopword removal + stemming
df['word_count'] = df['text_processed'].apply(lambda s: len(s.split()))
before = len(df)
df = df[df['word_count'] >= MIN_WORDS].copy().reset_index(drop=True)
print(f"After min-words filter ({MIN_WORDS}): removed {before - len(df):,} → {len(df):,} kept")

# De-duplicate identical processed texts within the same class
before = len(df)
df = df.drop_duplicates(subset=['text_processed', 'tag']).reset_index(drop=True)
print(f"After de-duplication:               removed {before - len(df):,} → {len(df):,} kept")

print(f"\nWord count stats after processing:")
print(df['word_count'].describe().to_string())

print(f"\nClass distribution after filtering:")
print(df['tag'].value_counts().to_string())




[3/6] Removing Russian stopwords and applying Snowball stemming...


stopwords+stem: 100%|██████████| 30207/30207 [00:17<00:00, 1766.99it/s]

After min-words filter (3): removed 198 → 30,009 kept
After de-duplication:               removed 115 → 29,894 kept

Word count stats after processing:
count    29894.000000
mean        35.179936
std         41.497891
min          3.000000
25%         11.000000
50%         21.000000
75%         41.000000
max        311.000000

Class distribution after filtering:
tag
депрессия         16519
тревожное р-во     5761
ОКР                2800
ПРЛ                2704
БАР                1376
шизофрения          734


In [27]:

# ══════════════════════════════════════════════════════════════════════════════
# Block 5 — Train / Test Split (80 / 20, stratified)
# ══════════════════════════════════════════════════════════════════════════════
print("\n[4/6] Splitting into train / test (80/20 stratified)...")

le = LabelEncoder()
y  = le.fit_transform(df['tag'].to_numpy())
X  = df['text_processed'].to_numpy(dtype=object)
NUM_CLASSES = len(le.classes_)

print(f"Classes ({NUM_CLASSES}): {list(le.classes_)}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = TEST_SIZE,
    random_state = RANDOM_STATE,
    stratify     = y,
)

print(f"Train: {len(y_train):,}  |  Test: {len(y_test):,}")
print("Train class counts:",
      dict(zip(le.classes_, np.bincount(y_train, minlength=NUM_CLASSES))))
print("Test  class counts:",
      dict(zip(le.classes_, np.bincount(y_test,  minlength=NUM_CLASSES))))




[4/6] Splitting into train / test (80/20 stratified)...
Classes (6): ['БАР', 'ОКР', 'ПРЛ', 'депрессия', 'тревожное р-во', 'шизофрения']
Train: 23,915  |  Test: 5,979
Train class counts: {'БАР': np.int64(1101), 'ОКР': np.int64(2240), 'ПРЛ': np.int64(2163), 'депрессия': np.int64(13215), 'тревожное р-во': np.int64(4609), 'шизофрения': np.int64(587)}
Test  class counts: {'БАР': np.int64(275), 'ОКР': np.int64(560), 'ПРЛ': np.int64(541), 'депрессия': np.int64(3304), 'тревожное р-во': np.int64(1152), 'шизофрения': np.int64(147)}


In [28]:

# ══════════════════════════════════════════════════════════════════════════════
# Block 6 — TF-IDF Feature Extraction
# ══════════════════════════════════════════════════════════════════════════════
print("\n[5/6] Extracting TF-IDF features...")

# Instantiated from TFIDF_PARAMS dict — change the dict in Block 1 to tune
tfidf = TfidfVectorizer(**TFIDF_PARAMS)

with tqdm(total=2, desc='TF-IDF', unit='step',
          bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}] {postfix}') as pbar:
    pbar.set_description('TF-IDF fit_transform (train)')
    X_train_tfidf = tfidf.fit_transform(X_train)
    pbar.set_postfix(shape=str(X_train_tfidf.shape))
    pbar.update(1)

    pbar.set_description('TF-IDF transform (test)')
    X_test_tfidf = tfidf.transform(X_test)
    pbar.set_postfix(shape=str(X_test_tfidf.shape))
    pbar.update(1)

print(f"TF-IDF matrix — train: {X_train_tfidf.shape}  |  test: {X_test_tfidf.shape}")
print(f"Vocabulary size: {len(tfidf.vocabulary_):,}")



[5/6] Extracting TF-IDF features...


TF-IDF transform (test): 100%|██████████| 2/2 [00:01] , shape=(5979, 27571)      

TF-IDF matrix — train: (23915, 27571)  |  test: (5979, 27571)
Vocabulary size: 27,571


In [29]:


# ══════════════════════════════════════════════════════════════════════════════
# Block 7 — Classification & Evaluation
# ══════════════════════════════════════════════════════════════════════════════
print("\n[6/6] Training classifiers and evaluating...")

TARGET_F1 = 0.60

results = {}  # label -> (f1_macro, accuracy, y_pred)

# ── Helper ─────────────────────────────────────────────────────────────────
def evaluate(label: str, clf, pbar: tqdm) -> None:
    """Fit clf on train TF-IDF, predict on test, store and print metrics."""
    pbar.set_description(f'fitting  {label}')
    clf.fit(X_train_tfidf, y_train)
    pbar.set_description(f'predict  {label}')
    y_pred = clf.predict(X_test_tfidf)
    f1     = f1_score(y_test, y_pred, average='macro')
    acc    = accuracy_score(y_test, y_pred)
    results[label] = (f1, acc, y_pred)
    pbar.set_postfix(F1=f'{f1:.4f}', Acc=f'{acc:.4f}')
    pbar.update(1)
    print(f"  {label:<35s} | F1={f1:.4f}  Acc={acc:.4f}")


# ── Model specs ────────────────────────────────────────────────────────────
model_specs = (
    [(f'LinearSVC  C={C}',
      LinearSVC(C=C, max_iter=8000, class_weight='balanced',
                random_state=RANDOM_STATE))
     for C in [0.1, 0.3, 1.0, 3.0, 10.0]]
    + [(f'LogisticReg C={C}',
        LogisticRegression(C=C, max_iter=4000, solver='lbfgs',
                           class_weight='balanced',
                           random_state=RANDOM_STATE, n_jobs=-1))
       for C in [1.0, 3.0, 10.0]]
)

with tqdm(total=len(model_specs), desc='classifiers', unit='model',
          bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}] {postfix}') as pbar:
    for label, clf in model_specs:
        evaluate(label, clf, pbar)

# ── Best model report ──────────────────────────────────────────────────────
best_label = max(results, key=lambda k: results[k][0])
f1_best, acc_best, y_pred_best = results[best_label]

print("\n" + "=" * 64)
print(f"BEST MODEL: {best_label}")
print("=" * 64)
print(f"Accuracy:      {acc_best:.4f}  ({acc_best * 100:.2f}%)")
print(f"F1 (macro):    {f1_best:.4f}")
print(f"F1 (weighted): {f1_score(y_test, y_pred_best, average='weighted'):.4f}")
print()
print("=== Per-class report ===")
print(classification_report(
    y_test, y_pred_best,
    target_names=list(le.classes_),
    digits=4,
))

print("=" * 64)
if f1_best >= TARGET_F1:
    print(f"✅ TARGET ACHIEVED: F1 macro = {f1_best:.4f} >= {TARGET_F1}")
else:
    print(f"❌ TARGET NOT MET:  F1 macro = {f1_best:.4f} < {TARGET_F1}")
print("=" * 64)

print("\n=== All Results Summary (sorted by F1 macro) ===")
for label, (f1, acc, _) in sorted(results.items(), key=lambda x: -x[1][0]):
    print(f"  F1={f1:.4f}  Acc={acc:.4f}  {label}")



[6/6] Training classifiers and evaluating...


fitting  LinearSVC  C=0.3:  12%|█▎        | 1/8 [00:00<00:00] , Acc=0.6906, F1=0.5781

  LinearSVC  C=0.1                    | F1=0.5781  Acc=0.6906


fitting  LinearSVC  C=1.0:  25%|██▌       | 2/8 [00:00<00:01] , Acc=0.6809, F1=0.5691

  LinearSVC  C=0.3                    | F1=0.5691  Acc=0.6809


...

  LogisticReg C=1.0                   | F1=0.5416  Acc=0.6245


fitting  LogisticReg C=10.0:  88%|████████▊ | 7/8 [00:05<00:01] , Acc=0.6469, F1=0.5561

  LogisticReg C=3.0                   | F1=0.5561  Acc=0.6469


predict  LogisticReg C=10.0: 100%|██████████| 8/8 [00:07<00:00] , Acc=0.6501, F1=0.5498

  LogisticReg C=10.0                  | F1=0.5498  Acc=0.6501

BEST MODEL: LinearSVC  C=0.1
Accuracy:      0.6906  (69.06%)
F1 (macro):    0.5781
F1 (weighted): 0.6850

=== Per-class report ===
                precision    recall  f1-score   support

           БАР     0.4840    0.4400    0.4610       275
           ОКР     0.6456    0.5661    0.6032       560
           ПРЛ     0.5593    0.4270    0.4843       541
     депрессия     0.7500    0.8163    0.7817      3304
тревожное р-во     0.6481    0.5868    0.6159      1152
    шизофрения     0.4677    0.5918    0.5225       147

      accuracy                         0.6906      5979
     macro avg     0.5925    0.5713    0.5781      5979
  weighted avg     0.6842    0.6906    0.6850      5979

❌ TARGET NOT MET:  F1 macro = 0.5781 < 0.6

=== All Results Summary (sorted by F1 macro) ===
  F1=0.5781  Acc=0.6906  LinearSVC  C=0.1
  F1=0.5691  Acc=0.6809  LinearSVC  C=0.3
  F1=0.5561  Acc=0.6469  LogisticReg C=3.0
  F1=0.5530  Acc=0.6625